# STOmics (Stereo-seq) Mouse Brain — cellbin analysis

数据: STOmics 小鼠脑组织 Stereo-seq cellbin 数据  
本 notebook 演示 trackcell 对 STO cellbin 数据的读入、分群分析和细胞边界展示。

## 数据格式说明

STOmics pipeline 产出几种数据格式：

| 目录 | 格式 | 说明 |
|---|---|---|
| `041.cellcut/` | `*.cellbin.gef` (HDF5) | 细胞分割数据，含 cellBorder 多边形轮廓 |
| `04.tissuecut/` | `*.tissue.gef` (HDF5) | 组织 bin 数据，按 500nm grid |
| `03.register/` | `*_regist.tif` | 配准后的 ssDNA 组织底图 |

**物理分辨率**: 1 GEF 坐标单位 = 500nm = 0.5μm. 图像像素与坐标 1:1 对齐。

In [ ]:
import scanpy as sc
import trackcell as tcl
import numpy as np
import matplotlib.pyplot as plt

sc.settings.set_figure_params(dpi=100, facecolor='white')
print(f"trackcell version: {tcl.__version__}")
print(f"scanpy version: {sc.__version__}")

## 1. 读入 cellbin GEF 数据

`tcl.io.read_sto_cellbin()` 可以传入 **文件路径** 或 **文件夹路径**（自动发现 `*cellbin*.gef`）。

如果同目录旁边有 `03.register/`，ssDNA 组织底图会被自动加载到 `adata.uns['spatial'][sample]['images']`。

In [ ]:
# 读入 cellbin GEF
adata = tcl.io.read_sto_cellbin(
    "SS200000135TL_D1.cellbin.gef",
    sample="mouse_brain"
)
adata

In [ ]:
# 查看 metadata 和 scalefactors
sample = "mouse_brain"
meta = adata.uns['spatial'][sample]['metadata']
sf = adata.uns['spatial'][sample]['scalefactors']

print("=== Metadata ===")
for k, v in meta.items():
    print(f"  {k}: {v}")
print(f"\n  Physical span: {(meta['coordinate_range']['x'][1] - meta['coordinate_range']['x'][0]) * meta['pixel_size_um']:.0f} μm")
print(f"  Cell count: {adata.n_obs:,}  |  Gene count: {adata.n_vars:,}")

## 2. QC 与预处理

In [ ]:
# 计算 QC metrics
sc.pp.calculate_qc_metrics(adata, inplace=True)
adata.layers["counts"] = adata.X.copy()

In [ ]:
# QC 小提琴图
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sc.pl.violin(adata, 'total_counts', ax=axes[0], show=False)
sc.pl.violin(adata, 'n_genes_by_counts', ax=axes[1], show=False)
sc.pl.violin(adata, 'gene_count', ax=axes[2], show=False)
fig.suptitle('QC metrics (before filtering)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# 过滤低质量细胞
print(f"Before filter: {adata.n_obs:,} cells, {adata.n_vars:,} genes")

sc.pp.filter_cells(adata, min_counts=200)
sc.pp.filter_cells(adata, min_genes=3)
sc.pp.filter_cells(adata, max_genes=2500)
sc.pp.filter_genes(adata, min_cells=3)

print(f"After filter:  {adata.n_obs:,} cells, {adata.n_vars:,} genes")

In [ ]:
# Normalization
adata.layers["counts"] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
adata.raw = adata.copy()

In [ ]:
# 高变基因
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat_v3')
sc.pl.highly_variable_genes(adata)
print(f"Highly variable genes: {adata.var.highly_variable.sum():,}")

In [ ]:
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata, max_value=10)
adata

## 3. 降维与聚类

In [ ]:
sc.tl.pca(adata, n_comps=30, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, n_pcs=30)
print("PCA done")

In [ ]:
sc.pp.neighbors(adata, n_pcs=15, n_neighbors=15)
sc.tl.umap(adata)
sc.tl.leiden(adata, resolution=0.8, key_added='leiden')

print(f"Clusters: {adata.obs['leiden'].nunique()}")
sc.pl.umap(adata, color='leiden', legend_loc='on data', legend_fontsize=8, title='Leiden clusters')

## 4. 空间可视化 — 细胞多边形

`tcl.pl.spatial_cell()` 用细胞多边形代替散点。如果加载了 ssDNA 底图，会自动叠加在背景。

In [ ]:
# 全局聚类图 — 细胞多边形着色
tcl.pl.spatial_cell(
    adata,
    color='leiden',
    library_id=sample,
    figsize=(10, 8),
    edges_width=0.3,
    alpha=0.85,
    legend=True,
    legend_fontsize=6
)
plt.title('Leiden clusters — cell polygons', fontsize=14)
plt.show()

### 基因表达空间图

挑几个 top marker genes 在空间上展示。

In [ ]:
# 计算 marker genes
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon', n_genes=5)

# 取每个 cluster 的 top-1 marker (去重)
top_markers = []
for cluster in adata.uns['rank_genes_groups']['names'].dtype.names:
    top_markers.append(adata.uns['rank_genes_groups']['names'][cluster][0])
top_markers = list(dict.fromkeys(top_markers))[:6]
print(f"Top markers: {top_markers}")

In [ ]:
# 多基因空间表达
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
for ax, gene in zip(axes.flat, top_markers):
    tcl.pl.spatial_cell(
        adata, color=gene, library_id=sample,
        figsize=None, ax=ax, show=False,
        edges_width=0.1, alpha=0.9,
        cmap='Reds', legend=False
    )
    ax.set_title(gene, fontsize=12)
plt.suptitle('Top marker gene expression', fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

## 5. 子区域 subset — 放大展示细胞边界

选取一块感兴趣区域 (ROI)，放大看细胞分割轮廓。

**注意**: subset 后必须调用 `tcl.io.sync_geometries_after_subset()` 同步 GeoDataFrame。

In [ ]:
def subset_region(adata, xlim, ylim, sample_id=sample):
    """按空间坐标 subset 一个矩形区域，同步 geometries。"""
    spatial = adata.obsm['spatial']
    mask = (
        (spatial[:, 0] >= xlim[0]) & (spatial[:, 0] <= xlim[1]) &
        (spatial[:, 1] >= ylim[0]) & (spatial[:, 1] <= ylim[1])
    )
    print(f"  Cells in ROI: {mask.sum():,} / {adata.n_obs:,}")

    sub = adata[mask].copy()
    # 同步 geometries GeoDataFrame
    tcl.io.sync_geometries_after_subset(sub, sample=sample_id)
    return sub

In [ ]:
# 查看坐标范围，选一个 ROI
meta = adata.uns['spatial'][sample]['metadata']
x_range = meta['coordinate_range']['x']
y_range = meta['coordinate_range']['y']
print(f"Full coordinate range: x={x_range}, y={y_range}")

# ROI: 取中间区域（约 1/4 面积）
x_mid = (x_range[0] + x_range[1]) // 2
y_mid = (y_range[0] + y_range[1]) // 2

xlim = [x_range[0], x_mid]
ylim = [y_mid, y_range[1]]

adata_roi = subset_region(adata, xlim, ylim)
print(f"ROI: {adata_roi.n_obs:,} cells × {adata_roi.n_vars:,} genes")

In [ ]:
# 放大区域 — 聚类着色
tcl.pl.spatial_cell(
    adata_roi,
    color='leiden',
    library_id=sample,
    figsize=(10, 8),
    edges_width=0.5,
    alpha=0.9,
    legend=True,
    legend_fontsize=5
)
plt.title(f'ROI: clusters [{xlim[0]}-{xlim[1]}, {ylim[0]}-{ylim[1]}]', fontsize=14)
plt.show()

In [ ]:
# 进一步缩小到 ~800×800 GEF 单位的 sub-ROI（≈ 400μm × 400μm）
x_center = (xlim[0] + xlim[1]) // 2
y_center = (ylim[0] + ylim[1]) // 2
zoom_half = 600  # GEF 坐标单位

sub_roi = subset_region(
    adata_roi,
    [x_center - zoom_half, x_center + zoom_half],
    [y_center - zoom_half, y_center + zoom_half]
)

In [ ]:
# 细胞边界细节 — edge_color 高亮聚类边界
tcl.pl.spatial_cell(
    sub_roi,
    color='leiden',
    edge_color='leiden',
    library_id=sample,
    figsize=(10, 8),
    edges_width=1.5,
    alpha=0.5,
    legend=True,
    legend_fontsize=6
)
plt.title(f'Zoom-in: cell boundaries ({zoom_half*2}×{zoom_half*2} units)', fontsize=14)
plt.show()

In [ ]:
# 单个 marker gene 在 zoom-in 区域的表达
gene_idx = 0
tcl.pl.spatial_cell(
    sub_roi,
    color=top_markers[gene_idx],
    library_id=sample,
    figsize=(10, 8),
    edges_width=0.3,
    edges_color='grey',
    alpha=0.9,
    cmap='Reds',
    legend=True
)
plt.title(f'Zoom-in: {top_markers[gene_idx]} expression', fontsize=14)
plt.show()

## 6. 保存 & 恢复

geometries 保存为 WKT 字符串以保证 h5ad 可序列化。读回时用 `tcl.io.restore_geometries()` 恢复 GeoDataFrame。

In [ ]:
# 保存
output_path = "mouse_brain_trackcell.h5ad"
adata.write_h5ad(output_path)
print(f"Saved to {output_path}")

# 读回 & restore
# adata2 = sc.read_h5ad(output_path)
# adata2 = tcl.io.restore_geometries(adata2, sample="mouse_brain")

## 附录：squarebin 数据读入

如果不需要细胞分割，使用 bin 级别分析：

In [ ]:
# squarebin 读入
bin_adata = tcl.io.read_sto_bin(
    "SS200000135TL_D1.tissue.gef",
    bin_size=50,          # 50 units = 25μm physical
    sample="mouse_brain"
)
print(f"Bins: {bin_adata.n_obs:,} × {bin_adata.n_vars:,} genes")
meta = bin_adata.uns['spatial']['mouse_brain']['metadata']
print(f"Physical bin size: {meta['bin_size']} × {meta['pixel_size_um']}μm = {meta['bin_size']*meta['pixel_size_um']:.0f}μm")

In [ ]:
# Bin 级别可视化
tcl.pl.spatial_bin(bin_adata, color='total_counts', library_id='mouse_brain', figsize=(10, 8))
plt.title('Square bin total counts', fontsize=14)
plt.show()